# MLFlow

[MLFlow documentation](https://mlflow.org/docs/latest/ml/tracking/)

In [1]:
# Match the MLflow version running in the docker deployment
%pip install mlflow==3.16.1 boto3


Note: you may need to restart the kernel to use updated packages.


Loading data and cleaning it up. [Load_wine](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) comes clean and ready to be used

In [2]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

data = load_wine()

X = data['data']
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model tracking using MLFlow. 

Here we train [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) and log
- Training parameters
- Run metrics
- Trained model inside the run

Model will be available within Experiment -> Run

In [3]:
import logging
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, log_loss
)

logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

# Connect to the local MLflow server (docker compose). Port comes from MLFLOW_PORT in .env
import os
os.environ["AWS_ACCESS_KEY_ID"] = "minio"          # MINIO_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123"   # MINIO_SECRET_ACCESS_KEY
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"  # MinIO, artifacts are uploaded from here

mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("demo")

model_info = None

# Model Tracking
with mlflow.start_run() as run:
    # pipeline.fit or model.fit()
    rf = RandomForestClassifier(max_depth=2, random_state=0)
    rf.fit(X_train, y_train)

    # Log model parameters
    mlflow.log_params(rf.get_params())

    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)

    # Log model metrics
    mlflow.log_metrics({
        "accuracy": rf.score(X_test, y_test),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "log_loss": log_loss(y_test, y_proba)
    })

    # Required to log model and then use them for inference
    signature = infer_signature(X_train, rf.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        rf,
        name="model",
        signature=signature,
        input_example=X_train[:5],
        # MLflow 3.16+ serializes with skops; RandomForest trees must be explicitly trusted
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    mlflow.set_tag("Training Info", "Basic RandomForestClassifier demo")

🏃 View run illustrious-pig-135 at: http://localhost:5001/#/experiments/1/runs/6eadc381bc18460aa5bb7195c044346a
🧪 View experiment at: http://localhost:5001/#/experiments/1


## Model registry and model alias in MLflow

We register the trained model in the MLflow Model Registry so we can use it in other Notebooks or in model serving


In [4]:
model_name : str = "rf_demo"

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,      # f"runs:/{run.info.run_id}/model"
    name=model_name
)
print(f"Registered {registered_model.name} version {registered_model.version}")

Registered model 'rf_demo' already exists. Creating a new version of this model...
2026/09/26 12:09:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: rf_demo, version 2


Registered rf_demo version 2


Created version '2' of model 'rf_demo'.


We assign the newest trained model an alias so we can easily identify the latest mlflow experiment run. By common convention, we use `champion` as our latest model alias.

In [5]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered_model.version
)